In [0]:
%sql
CREATE OR REPLACE VIEW olist_catalog.olist_semantic.marketing_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: olist_catalog.olist_gold.fact_marketing
  comment: Marketing campaign performance metrics with spend analysis and channel effectiveness
  
  joins:
    - name: sellers
      source: olist_catalog.olist_gold.dim_sellers
      on: source.seller_key = sellers.seller_id
    - name: time_dim
      source: olist_catalog.olist_gold.dim_time
      on: source.campaign_date_key = time_dim.date_key
  
  dimensions:
    - name: campaign_year
      expr: source.campaign_year
      display_name: Campaign Year
      comment: Year when the marketing campaign ran
      synonyms:
        - year
    - name: campaign_month
      expr: time_dim.month_name
      display_name: Campaign Month
      comment: Month when the marketing campaign ran
      synonyms:
        - month
    - name: year_month
      expr: time_dim.year_month
      display_name: Year-Month
      comment: Combined year and month for time series analysis
      synonyms:
        - period
        - time period
    - name: channel
      expr: source.channel
      display_name: Marketing Channel
      comment: Marketing channel used for the campaign
      synonyms:
        - channel type
        - marketing channel
    - name: seller_region
      expr: sellers.seller_region
      display_name: Seller Region
      comment: Geographic region of the seller
      synonyms:
        - region
        - seller location
    - name: spend_category
      expr: source.spend_category
      display_name: Spend Category
      comment: Categorization of advertising spend level
      synonyms:
        - budget category
    - name: discount_category
      expr: source.discount_category
      display_name: Discount Category
      comment: Categorization of discount level offered
      synonyms:
        - promotion level
  
  measures:
    - name: total_ad_spend
      expr: SUM(source.ad_spend)
      display_name: Total Ad Spend
      comment: Total advertising spend across campaigns
      format:
        type: currency
        currency_code: BRL
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - ad spend
        - advertising cost
        - marketing spend
    - name: campaign_count
      expr: COUNT(1)
      display_name: Campaign Count
      comment: Number of marketing campaigns
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - campaigns
        - number of campaigns
    - name: total_quantity
      expr: SUM(source.quantity)
      display_name: Total Quantity
      comment: Total quantity promoted in campaigns
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - quantity
        - items promoted
    - name: avg_discount_percent
      expr: AVG(source.discount_percent)
      display_name: Average Discount %
      comment: Average discount percentage offered
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - discount
        - average discount
    - name: avg_ad_spend
      expr: AVG(source.ad_spend)
      display_name: Average Ad Spend
      comment: Average advertising spend per campaign
      format:
        type: currency
        currency_code: BRL
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - avg spend
        - cost per campaign
    - name: cost_per_unit
      expr: SUM(source.ad_spend) / NULLIF(SUM(source.quantity), 0)
      display_name: Cost per Unit
      comment: Advertising cost per unit quantity
      format:
        type: currency
        currency_code: BRL
        decimal_places:
          type: exact
          places: 2
      synonyms:
        - unit cost
        - CPA
$$